# Notebook 3 — Rare Event Estimation

Safety-critical events (near-misses, emergency stops) are rare by definition.
This notebook uses **Extreme Value Theory (EVT)** to estimate the probability
of dangerous events far beyond what has been directly observed.

Methods covered:
- Block Maxima / GEV distribution fitting
- Peaks-Over-Threshold (POT) / GPD fitting
- Return-level plots (e.g., 1-in-1000 frame TTC)
- Physical rare-event detector (near-miss, hard braking, swerve)

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.waymo_loader import load_dataset, extract_trajectories
from src.metrics.safety_metrics import batch_ttc, nearest_agent_distances
from src.rare_events.evt_analysis import (
    fit_gev, extract_block_minima, fit_gpd,
    select_threshold_mean_excess, rare_event_summary
)
from src.rare_events.rare_event_detector import run_all_detectors
from src.visualization.plots import (
    plot_evt_fit, plot_mean_excess, plot_rare_events_timeline
)

In [2]:
dataset = load_dataset('../data', max_segments=5)
trajectories = extract_trajectories(dataset['lidar_box'])
ttc_df = batch_ttc(trajectories)
ttc_series = ttc_df['min_ttc']
print(f'TTC observations: {len(ttc_series):,}')
print(f'Finite TTC frames: {np.isfinite(ttc_series).sum():,}')

TTC observations: 986
Finite TTC frames: 986


/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)


## Block Maxima — GEV Fitting

In [3]:
block_minima = extract_block_minima(ttc_series, block_size=100)
print(f'Number of blocks: {len(block_minima)}')
print(f'Block minima range: [{block_minima.min():.3f}, {block_minima.max():.3f}] s')

# Negate to convert minima → maxima for GEV
gev = fit_gev(-block_minima)
print(f'\nGEV fit:')
print(f'  xi (shape) = {gev.xi:.4f}')
print(f'  mu (location) = {gev.mu:.4f}')
print(f'  sigma (scale) = {gev.sigma:.4f}')
print(f'  AIC = {gev.aic:.2f}')

Number of blocks: 9
Block minima range: [0.000, 13.615] s

GEV fit:
  xi (shape) = -1.0459
  mu (location) = -0.6659
  sigma (scale) = 0.6964
  AIC = 13.18


In [4]:
fig = plot_evt_fit(ttc_series, gev_fit=gev, return_periods=[10, 100, 1000])
plt.show()

/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/rare_events/evt_analysis.py:51: RuntimeWarning: divide by zero encountered in log
  return self.mu + self.sigma * ((-np.log(p)) ** (-self.xi) - 1) / self.xi


/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27713/1787332022.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Peaks-Over-Threshold — GPD Fitting

In [5]:
finite_ttc = ttc_series[np.isfinite(ttc_series)].values
thresholds, mean_exc, suggested = select_threshold_mean_excess(finite_ttc)
print(f'Suggested threshold: {suggested:.3f} s')

fig = plot_mean_excess(thresholds, mean_exc, suggested)
plt.show()

Suggested threshold: 79.151 s


/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27713/11858254.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# GPD on the LOW tail of TTC: invert (high 1/TTC = low TTC = dangerous)
# Exclude zero-TTC frames (agents already inside safety box) before inverting
pos_ttc = finite_ttc[finite_ttc > 0.05]
inv_ttc = 1.0 / pos_ttc
gpd = fit_gpd(inv_ttc, threshold=float(np.quantile(inv_ttc, 0.85)))
print(f'GPD fit:')
print(f'  xi (shape) = {gpd.xi:.4f}')
print(f'  sigma (scale) = {gpd.sigma:.4f}')
print(f'  threshold = {gpd.threshold:.4f}')
print(f'  exceedances = {gpd.n_exceedances} / {gpd.n_total}')

GPD fit:
  xi (shape) = -0.2703
  sigma (scale) = 2.3256
  threshold = 3.2461
  exceedances = 87 / 577


## Return Level Summary Table

In [7]:
summary = rare_event_summary(ttc_series, return_periods=[10, 100, 1000])
print('Estimated TTC return levels (how low TTC gets at each return period):')
summary.pivot(index='return_period', columns='method', values='ttc_return_level_s').round(3)

Estimated TTC return levels (how low TTC gets at each return period):


method,GEV,GPD
return_period,,
NaN,NaN,NaN
10.0,NaN,0.096
100.0,NaN,0.090
1000.0,NaN,0.087


## Physical Rare-Event Detection

In [8]:
events_df = run_all_detectors(trajectories, ttc_df=ttc_df)
print(f'Total rare events detected: {len(events_df)}')
if not events_df.empty:
    print(events_df['event_type'].value_counts())
    print(events_df['severity'].value_counts())

Total rare events detected: 51
event_type
sudden_swerve              38
near_miss                   6
pedestrian_encroachment     6
hard_braking                1
Name: count, dtype: int64
severity
warning     39
critical    12
Name: count, dtype: int64


In [9]:
fig = plot_rare_events_timeline(events_df)
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_27713/298090525.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
